In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import DBSCAN, AgglomerativeClustering, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score 
import umap
from sklearn.metrics import silhouette_samples
from scipy.stats import f_oneway

# Paths
data_folder = 'data'
file_name = 'UNSW-NB15_1.csv'
header_file = 'NUSW-NB15_features.csv'
output_folder = 'output'
plots_folder = os.path.join(output_folder, 'plots')
os.makedirs(output_folder, exist_ok=True)
os.makedirs(plots_folder, exist_ok=True)

# Load header 
header_df = pd.read_csv(os.path.join(data_folder, header_file), encoding='ISO-8859-1')
header = header_df['Name'].tolist()

# Load data file without headers
df = pd.read_csv(os.path.join(data_folder, file_name), header=None)

# Merge header
df.columns = header
print("Data shape =", df.shape) 
df_full=df.copy()
# Drop response variables
drop_cols = ['Label', 'attack_cat']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

df.to_csv(os.path.join(output_folder, 'df.csv'), index=False)
print("df saved")
print(f'df shape =',df.shape) 

# Feature Engineering
df['packet_rate'] = df['Spkts'] / (df['dur'] + 1e-5)
df['byte_ratio'] = df['sbytes'] / (df['dbytes'] + 1e-5)
df['flag_activity'] = df['state'].isin(['fin', 'syn', 'rst', 'psh', 'ack', 'urg']).astype(int)

print("After feature engineering:", df.shape)

# Select numeric features
df_numeric = df.select_dtypes(include=[np.number])
print("Numeric features shape:", df_numeric.shape)

# Drop highly correlated features
corr_matrix = df_numeric.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = {upper.columns[i] 
           for i in range(len(upper.columns)) 
           for j in range(i) 
           if upper.iloc[j, i] > 0.95}

print("Dropped due to high correlation:", to_drop)

df_numeric = df_numeric.drop(columns=list(to_drop))
print("After dropping correlated features:", df_numeric.shape)

# Drop low-variance features
selector = VarianceThreshold(threshold=0.01)
df_selected = selector.fit_transform(df_numeric)
selected_features = df_numeric.columns[selector.get_support()]

print("After variance threshold:", df_selected.shape)

# Impute missing values
imputer = SimpleImputer(strategy='mean')
df_imputed = imputer.fit_transform(df_selected)

# Standardize data for PCA
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_imputed)
print("Final scaled data shape:", df_scaled.shape)

# PCA---------------------------------------
pca = PCA(n_components=0.85)
df_pca = pca.fit_transform(df_scaled)
print("Number of PCA components explaining 90% variance:", pca.n_components_)
# Since our dataset is large, we take only the first 10 PCs
n_pc = min(5, df_pca.shape[1])
df_pca = df_pca[:, :n_pc]
print(f"Using first {n_pc} PCs for clustering and plotting.")
# Compute loadings & arrow lengths
loadings = pca.components_.T  # shape: (n_features, n_components)
arrow_lengths = np.linalg.norm(loadings[:, :2], axis=1)  # Use only PC1 & PC2 for visualization
top_indices = np.argsort(arrow_lengths)[-3:]  # top 3 features contributing to PC1 & PC2
top_features = [selected_features[i] for i in top_indices]
print("Top features for arrows:", top_features)

# Plot PCA scatter with arrows
print('\nPCA ------------------------')
plt.figure(figsize=(10,8))
plt.scatter(df_pca[:,0], df_pca[:,1], s=5, alpha=0.6, c='black')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Scatter (PC1 vs PC2) with Top Feature Directions")
plt.grid(True)

# Scale arrows automatically based on data spread
x_max = np.max(np.abs(df_pca[:,0]))
y_max = np.max(np.abs(df_pca[:,1]))
arrow_scale = max(x_max, y_max) * 0.8

for i in top_indices:
    x, y = loadings[i, 0], loadings[i, 1]
    plt.arrow(0, 0, x*arrow_scale, y*arrow_scale, color='red', alpha=0.8, head_width=0.1)
    plt.text(x*arrow_scale*1.1, y*arrow_scale*1.1, selected_features[i], color='darkred', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'pca_plot.png'))
print('pca plot saved')
plt.close()

# PVE Plot
explained_var = pca.explained_variance_ratio_
cum_var = np.cumsum(explained_var)

plt.figure(figsize=(8,5))
plt.bar(range(1, len(explained_var)+1), explained_var, alpha=0.6, label='Individual PVE')
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o', color='red', label='Cumulative PVE')
plt.xlabel("Principal Component")
plt.ylabel("Proportion of Variance Explained")
plt.title("PVE (Proportion of Variance Explained) Plot")
plt.xticks(range(1, len(explained_var)+1))
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'pve_plot.png'))
print('pve plot saved')
plt.close()

# UMAP on original scaled data
print('\nUMAP-------------------------')
sample_size = 10000
df_pca_sample = df_pca[:sample_size, :]  
# UMAP on PCA-reduced data 
umap_model = umap.UMAP(n_neighbors=5, min_dist=0.3, n_epochs=200, random_state=42)
df_umap = umap_model.fit_transform(df_pca_sample)
print("UMAP embedding shape:", df_umap.shape)

# Plot UMAP
plt.figure(figsize=(10,8))
plt.scatter(df_umap[:,0], df_umap[:,1], s=5, alpha=0.6, c='black')
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.title("UMAP Embedding")
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'umap_plot.png'))
print('umap plot saved')
plt.close()

# clustering
print('\nClustering-----------------------')
sample_size = 10000
X = df_pca[:sample_size, :]

# DBSCAN hyperparameter tuning
eps_values = [0.1, 0.2, 0.3, 0.4, 0.5]
min_samples_values = [5, 10, 15]

best_dbscan_score = -1
best_dbscan_params = None
best_dbscan_labels = None

print("DBSCAN")
for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X)
        # skip if single cluster or mostly noise
        if len(set(labels)) <= 1 or (len(labels) - list(labels).count(-1)) < 2: 
            continue
        score = silhouette_score(X, labels)
        if score > best_dbscan_score:
            best_dbscan_score = score
            best_dbscan_params = (eps, min_samples)
            best_dbscan_labels = labels

# Outliers are DBSCAN points with label -1
dbscan_outliers = best_dbscan_labels == -1

print(f"DBSCAN Best Params: {best_dbscan_params}")
print(f"Silhouette: {best_dbscan_score:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X, best_dbscan_labels):.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X, best_dbscan_labels):.4f}")
n_clusters = len(set(best_dbscan_labels)) - (1 if -1 in best_dbscan_labels else 0)
print("Number of clusters (excluding noise):", n_clusters)
print(f"Number of outliers: {np.sum(dbscan_outliers)}")

# K-Means tuning
n_clusters_list = [3, 4, 5, 6, 7]
best_kmeans_score = -1
best_kmeans_n = None
best_kmeans_labels = None

print("\nK-Means")
for n_clusters in n_clusters_list:
    kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=500)
    labels = kmeans.fit_predict(X)
    score = silhouette_score(X, labels)
    if score > best_kmeans_score:
        best_kmeans_score = score
        best_kmeans_n = n_clusters
        best_kmeans_labels = labels

# Function to detect outliers based on distance from cluster centroid
def detect_outliers(X, labels, std_thresh=2.0):
    outlier_mask = np.zeros(len(X), dtype=bool)
    unique_labels = np.unique(labels)
    for lbl in unique_labels:
        if lbl == -1:
            continue
        cluster_points = X[labels == lbl]
        centroid = cluster_points.mean(axis=0)
        distances = np.linalg.norm(cluster_points - centroid, axis=1)
        threshold = distances.mean() + std_thresh * distances.std()
        outlier_mask[labels == lbl] = distances > threshold
    return outlier_mask

kmeans_outliers = detect_outliers(X, best_kmeans_labels)

print(f"K-Means Best n_clusters: {best_kmeans_n}")
print(f"Silhouette: {best_kmeans_score:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X, best_kmeans_labels):.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X, best_kmeans_labels):.4f}")
print(f"Number of outliers: {np.sum(kmeans_outliers)}")

# Hierarchical Clustering tuning
linkages = ['ward', 'average', 'complete']
best_hier_score = -1
best_hier_params = None
best_hier_labels = None

print("\nHierarchical Clustering")
for n_clusters in n_clusters_list:
    for linkage in linkages:
        try:
            hier = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
            labels = hier.fit_predict(X)
            score = silhouette_score(X, labels)
            if score > best_hier_score:
                best_hier_score = score
                best_hier_params = (n_clusters, linkage)
                best_hier_labels = labels
        except:
            continue

hier_outliers = detect_outliers(X, best_hier_labels)

print(f"Hierarchical Best Params: {best_hier_params}")
print(f"Silhouette: {best_hier_score:.4f}")
print(f"Davies-Bouldin Index: {davies_bouldin_score(X, best_hier_labels):.4f}")
print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X, best_hier_labels):.4f}")
print(f"Number of outliers: {np.sum(hier_outliers)}")

# Combined Plot
def combined_cluster_plots(X, db_labels, km_labels, hier_labels,plots_folder):
    titles = ["DBSCAN Clusters", "K-Means Clusters", "Hierarchical Clusters"]
    label_sets = [db_labels, km_labels, hier_labels]

    plt.figure(figsize=(18, 5))

    for i, (labels, title) in enumerate(zip(label_sets, titles), 1):
        plt.subplot(1, 3, i)
        unique_labels = np.unique(labels)
        palette = sns.color_palette('tab10', len(unique_labels))
        for j, lbl in enumerate(unique_labels):
            plt.scatter(X[labels==lbl, 0], X[labels==lbl, 1], 
                        s=10, alpha=0.6, color=palette[j % 10], label=str(lbl))
        plt.title(title)
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(plots_folder, 'cluster_plot.png'))
    print('cluster plot saved')
    plt.close()

def combined_outlier_plots(X, db_out, km_out, hier_out,plots_folder):
    titles = ["DBSCAN Outliers", "K-Means Outliers", "Hierarchical Outliers"]
    outlier_masks = [db_out, km_out, hier_out]

    plt.figure(figsize=(18, 5))

    for i, (mask, title) in enumerate(zip(outlier_masks, titles), 1):
        plt.subplot(1, 3, i)
        plt.scatter(X[:,0], X[:,1], s=5, alpha=0.3, color='blue', label="Normal")
        plt.scatter(X[mask,0], X[mask,1], s=20, alpha=0.8, color='red', label="Outlier")
        plt.title(title)
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(plots_folder, 'outliar_plot.png'))
    print('outliar plot saved')
    plt.close()

# Combined cluster plots
combined_cluster_plots(X, best_dbscan_labels, best_kmeans_labels, best_hier_labels,plots_folder)

# Combined outlier plots
combined_outlier_plots(X, dbscan_outliers, kmeans_outliers, hier_outliers,plots_folder)

# Actual PCA data plot 
plt.figure(figsize=(10,8))
plt.scatter(X[:,0], X[:,1], s=5, alpha=0.6, color='black')
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Sample Data Points")
plt.grid(True)
plt.savefig(os.path.join(plots_folder, 'data_plot.png'))
print('data plot saved')
plt.close()

# Best clustering method based on Silhouette score
print('\nBest model----------------------')

# Store silhouette scores in a dictionary
silhouette_scores = {
    "DBSCAN": best_dbscan_score,
    "KMeans": best_kmeans_score,
    "Hierarchical": best_hier_score
}

# Find the method with the highest silhouette score
best_method_name = max(silhouette_scores, key=silhouette_scores.get)
best_score = silhouette_scores[best_method_name]

# Select the corresponding labels
if best_method_name == "DBSCAN":
    best_labels = best_dbscan_labels
elif best_method_name == "KMeans":
    best_labels = best_kmeans_labels
else:
    best_labels = best_hier_labels

print(f"Best Clustering Method: {best_method_name}")
print(f"Silhouette Score: {best_score:.4f}")


# Best model (Hierarchical) labels
hier_labels = best_hier_labels

# DBSCAN labels
dbscan_labels = best_dbscan_labels

# K-Means labels
kmeans_labels = best_kmeans_labels

# Compare Hierarchical vs DBSCAN
ari_hier_dbscan = adjusted_rand_score(hier_labels, dbscan_labels)
nmi_hier_dbscan = normalized_mutual_info_score(hier_labels, dbscan_labels)
print('\nHierarchical vs DBSCAN')
print(f"ARI: {ari_hier_dbscan:.4f}")
print(f"NMI: {nmi_hier_dbscan:.4f}")

# Compare Hierarchical vs K-Means
ari_hier_kmeans = adjusted_rand_score(hier_labels, kmeans_labels)
nmi_hier_kmeans = normalized_mutual_info_score(hier_labels, kmeans_labels)
print('\nHierarchical vs K-Means')
print(f"ARI: {ari_hier_kmeans:.4f}")
print(f"NMI: {nmi_hier_kmeans:.4f}")



C:\Users\tanuja varma\AppData\Local\Temp\ipykernel_12976\4244688.py:31: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(data_folder, file_name), header=None)


Data shape = (700001, 49)
df saved
df shape = (700001, 47)
After feature engineering: (700001, 50)
Numeric features shape: (700001, 43)
Dropped due to high correlation: {'Ltime', 'dloss', 'dwin', 'Dpkts'}
After dropping correlated features: (700001, 39)
After variance threshold: (700001, 34)
Final scaled data shape: (700001, 34)
Number of PCA components explaining 90% variance: 17
Using first 5 PCs for clustering and plotting.
Top features for arrows: ['ct_dst_sport_ltm', 'ct_src_dport_ltm', 'ct_dst_src_ltm']

PCA ------------------------
pca plot saved
pve plot saved

UMAP-------------------------


c:\Python312\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embedding shape: (10000, 2)
umap plot saved

Clustering-----------------------
DBSCAN
DBSCAN Best Params: (0.5, 15)
Silhouette: 0.2042
Davies-Bouldin Index: 1.2069
Calinski-Harabasz Index: 512.3022
Number of clusters (excluding noise): 23
Number of outliers: 1101

K-Means
K-Means Best n_clusters: 7
Silhouette: 0.3957
Davies-Bouldin Index: 0.9372
Calinski-Harabasz Index: 3695.8139
Number of outliers: 475

Hierarchical Clustering
Hierarchical Best Params: (3, 'average')
Silhouette: 0.7801
Davies-Bouldin Index: 0.2388
Calinski-Harabasz Index: 315.2509
Number of outliers: 484
cluster plot saved
outliar plot saved
data plot saved

Best model----------------------
Best Clustering Method: Hierarchical
Silhouette Score: 0.7801

Hierarchical vs DBSCAN
ARI: 0.0015
NMI: 0.0040

Hierarchical vs K-Means
ARI: 0.0009
NMI: 0.0045


In [2]:
#summary table
print("\nSummary table ----------------------")

# Function to compute key metrics safely
def compute_metrics(X, labels):
    # If only one cluster, metrics cannot be computed
    if len(np.unique(labels)) < 2:
        return {
            "Silhouette": np.nan,
            "Davies-Bouldin": np.nan,
            "Calinski-Harabasz": np.nan,
            "Clusters": len(np.unique(labels)),
            "Noise/Outliers": np.sum(labels == -1)
        }

    return {
        "Silhouette": silhouette_score(X, labels),
        "Davies-Bouldin": davies_bouldin_score(X, labels),
        "Calinski-Harabasz": calinski_harabasz_score(X, labels),
        "Clusters": len(np.unique(labels)),
        "Noise/Outliers": np.sum(labels == -1)
    }

# Compute metrics for each clustering algorithm
stats_dbscan = compute_metrics(X, best_dbscan_labels)
stats_kmeans = compute_metrics(X, best_kmeans_labels)
stats_hier   = compute_metrics(X, best_hier_labels)

# Combine results into a table
stats_table = pd.DataFrame([
    ["DBSCAN", stats_dbscan["Silhouette"], stats_dbscan["Davies-Bouldin"],
     stats_dbscan["Calinski-Harabasz"], stats_dbscan["Clusters"], stats_dbscan["Noise/Outliers"]],

    ["K-Means", stats_kmeans["Silhouette"], stats_kmeans["Davies-Bouldin"],
     stats_kmeans["Calinski-Harabasz"], stats_kmeans["Clusters"], np.sum(kmeans_outliers)],

    ["Hierarchical", stats_hier["Silhouette"], stats_hier["Davies-Bouldin"],
     stats_hier["Calinski-Harabasz"], stats_hier["Clusters"], np.sum(hier_outliers)]
],

columns=["Method", "Silhouette", "Davies-Bouldin", "Calinski-Harabasz", "Num Clusters", "Outliers"])

print(stats_table)

# Plot Silhouette 
plt.figure(figsize=(8, 5))
sns.barplot(data=stats_table, x="Method", y="Silhouette")
plt.title("Silhouette Score Comparison")
plt.grid(axis="y")
plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'silhouette_comparison.png'))
print("Silhouette comparison plot saved")
plt.close()



Summary table ----------------------


         Method  Silhouette  Davies-Bouldin  Calinski-Harabasz  Num Clusters  \
0        DBSCAN    0.204235        1.206923         512.302229            24   
1       K-Means    0.395743        0.937228        3695.813858             7   
2  Hierarchical    0.780084        0.238761         315.250893             3   

   Outliers  
0      1101  
1       475  
2       484  
Silhouette comparison plot saved


In [3]:
#anova test
print('\nANOVA test on Silhouette Scores---------------------')
# Per-sample silhouette scores
sil_samples_dbscan = silhouette_samples(X, best_dbscan_labels)
sil_samples_kmeans = silhouette_samples(X, best_kmeans_labels)
sil_samples_hier = silhouette_samples(X, best_hier_labels)

# ANOVA Test for Silhouette scores
f_stat, p_value = f_oneway(sil_samples_dbscan, sil_samples_kmeans, sil_samples_hier)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("Result: Significant difference exists between clustering methods (p < 0.05)")
else:
    print("Result: No significant difference (p >= 0.05)")

# ARI and NMI relative to best model (Hierarchical)
ari_hier_dbscan = adjusted_rand_score(best_hier_labels, best_dbscan_labels)
ari_hier_kmeans = adjusted_rand_score(best_hier_labels, best_kmeans_labels)

nmi_hier_dbscan = normalized_mutual_info_score(best_hier_labels, best_dbscan_labels)
nmi_hier_kmeans = normalized_mutual_info_score(best_hier_labels, best_kmeans_labels)

# 4. Validation Table (highlight best model)
validation_table = pd.DataFrame({
    "Method": ["DBSCAN", "KMeans", "Hierarchical"],
    "Silhouette": [best_dbscan_score, best_kmeans_score, best_hier_score],
    "Best": ["", "", "<-- Best"]
})

print("\nSilhouette Score:")
print(validation_table)

print("\nARI Relative to Hierarchical:")
print(f"Hierarchical vs DBSCAN: {ari_hier_dbscan:.4f}")
print(f"Hierarchical vs KMeans: {ari_hier_kmeans:.4f}")

print("\nNMI Relative to Hierarchical:")
print(f"Hierarchical vs DBSCAN: {nmi_hier_dbscan:.4f}")
print(f"Hierarchical vs KMeans: {nmi_hier_kmeans:.4f}")




ANOVA test on Silhouette Scores---------------------
F-statistic: 9972.7598
P-value: 0.0000e+00
Result: Significant difference exists between clustering methods (p < 0.05)

Silhouette Score:
         Method  Silhouette      Best
0        DBSCAN    0.204235          
1        KMeans    0.395743          
2  Hierarchical    0.780084  <-- Best

ARI Relative to Hierarchical:
Hierarchical vs DBSCAN: 0.0015
Hierarchical vs KMeans: 0.0009

NMI Relative to Hierarchical:
Hierarchical vs DBSCAN: 0.0040
Hierarchical vs KMeans: 0.0045


In [4]:
# Temporal analysis
print('\nTemporal Analysis--------------------')
df_scaled_df = pd.DataFrame(df_scaled, columns=selected_features)
df_scaled_df['Stime'] = df['Stime'].values

# Convert Unix time to datetime
df_scaled_df['Stime'] = pd.to_datetime(df_scaled_df['Stime'], unit='s')
df_scaled_df.sort_values('Stime', inplace=True)

features = [col for col in df_scaled_df.columns if col != 'Stime']

# Hourly clustering
batch_size = '1H'
anomaly_log = []
cluster_patterns = []
summary_list = []

hourly_batches = list(df_scaled_df.groupby(pd.Grouper(key='Stime', freq=batch_size)))
n_batches = len(hourly_batches)

# Combined PCA cluster plots 
ncols = 3
nrows = (n_batches + ncols - 1) // ncols
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 5*nrows))
axes = axes.flatten()

for idx, (batch_time, batch_df) in enumerate(hourly_batches):
    if len(batch_df) < 2:
        continue
    
    X_batch = batch_df[features].values
    if len(X_batch) > 10000:
        X_batch = X_batch[:10000]
        batch_df = batch_df.iloc[:10000]

    # Agglomerative clustering
    hier = AgglomerativeClustering(n_clusters=3, linkage='average')
    labels = hier.fit_predict(X_batch)

    # Outlier detection
    outlier_mask = np.zeros(len(X_batch), dtype=bool)
    for lbl in np.unique(labels):
        cluster_points = X_batch[labels == lbl]
        centroid = cluster_points.mean(axis=0)
        distances = np.linalg.norm(cluster_points - centroid, axis=1)
        threshold = distances.mean() + 2 * distances.std()
        outlier_mask[labels == lbl] = distances > threshold

    # Log anomalies
    anomaly_log.append(pd.DataFrame({
        'Stime': batch_df['Stime'],
        'is_anomaly': outlier_mask
    }))

    # Log cluster sizes
    counts_per_cluster = {f'Cluster {lbl}': np.sum(labels == lbl) for lbl in np.unique(labels)}
    cluster_patterns.append({'batch_time': batch_time, 'cluster_counts': counts_per_cluster})
    
    # Add to summary table
    summary_row = {'Hour': batch_time.hour, 'Num_Anomalies': np.sum(outlier_mask)}
    summary_row.update(counts_per_cluster)
    summary_list.append(summary_row)

    # PCA plot
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_batch)

    ax = axes[idx]
    unique_clusters = np.unique(labels)
    colors = sns.color_palette('tab10', len(unique_clusters))

    for i, cluster_id in enumerate(unique_clusters):
        cluster_mask = (labels == cluster_id) & (~outlier_mask)
        ax.scatter(X_pca[cluster_mask,0], X_pca[cluster_mask,1],
                   color=colors[i], s=10, label=f'Cluster {cluster_id} ({counts_per_cluster[f"Cluster {cluster_id}"]} pts)')

    ax.scatter(X_pca[outlier_mask,0], X_pca[outlier_mask,1],
               color='red', s=20, alpha=0.7, label=f'Outliers ({np.sum(outlier_mask)})')

    ax.set_title(f'Hour {batch_time.hour}:00')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.grid(True)
    ax.legend(loc='upper right', fontsize='small')

# Hide empty subplots
for j in range(idx+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'time_cluster_plot.png'))
print('time_cluster_plot saved')
plt.close()

# Anomalies per hour (bar plot)
anomaly_df = pd.concat(anomaly_log, ignore_index=True)
all_hours = list(range(11, 20))
anomaly_df['hour'] = anomaly_df['Stime'].dt.hour
hourly_anomalies = anomaly_df.groupby('hour')['is_anomaly'].sum().reindex(all_hours, fill_value=0)

plt.figure(figsize=(10,6))
sns.barplot(x=hourly_anomalies.index, y=hourly_anomalies.values, color='red')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Anomalies')
plt.title('Temporal Analysis: Anomalies per Hour')
plt.grid(True)
plt.savefig(os.path.join(plots_folder, 'anomaly_time.png'))
print('anomaly_time saved')
plt.close()

# Cluster size over time (line plot)
cluster_df = pd.DataFrame([
    {'batch_time': cp['batch_time'], 'cluster': c, 'count': n}
    for cp in cluster_patterns for c, n in cp['cluster_counts'].items()
])

plt.figure(figsize=(12,6))
sns.lineplot(data=cluster_df, x='batch_time', y='count', hue='cluster', marker='o')
plt.xlabel('Time')
plt.ylabel('Number of Points in Cluster')
plt.title('Hierarchical Cluster Patterns Over Time')
plt.grid(True)
plt.savefig(os.path.join(plots_folder, 'cluster_size_time.png'))
print('cluster_size_time saved')
plt.close()

# Combined individual cluster size vs time 
cluster_df.sort_values('batch_time', inplace=True)
clusters = sorted(cluster_df['cluster'].unique())

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 15), sharex=True)

for i, cluster_id in enumerate(clusters):
    ax = axes[i]
    data = cluster_df[cluster_df['cluster'] == cluster_id]
    ax.plot(data['batch_time'], data['count'], marker='o', label=f'Cluster {cluster_id}', color=sns.color_palette('tab10')[i])
    ax.set_ylabel('Number of Points')
    ax.set_title(f'Cluster {cluster_id} Size Over Time')
    ax.grid(True)
    ax.legend()

axes[-1].set_xlabel('Time')
plt.tight_layout()
plt.savefig(os.path.join(plots_folder, 'cluster_size_time1.png'))
print('cluster_size_time1 saved')
plt.close()

# Summary table
summary_df = pd.DataFrame(summary_list).fillna(0).astype(int)
print('\nsummary table:')
print(summary_df)



Temporal Analysis--------------------


C:\Users\tanuja varma\AppData\Local\Temp\ipykernel_12976\2320395747.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_batches = list(df_scaled_df.groupby(pd.Grouper(key='Stime', freq=batch_size)))


time_cluster_plot saved
anomaly_time saved
cluster_size_time saved
cluster_size_time1 saved

summary table:
   Hour  Num_Anomalies  Cluster 0  Cluster 1  Cluster 2
0    11            358       9997          1          2
1    12            209       9992          6          2
2    13            215       9996          2          2
3    14            412       9954         22         24
4    15            488       9986         10          4
5    16            403       9946         22         32
6    17            422       9991          8          1
7    18            403       9977         22          1
8    19            394       9989          2          9
